In [2]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom
from jax_sbgeom.flux_surfaces import ToroidalExtent
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh
import jax.numpy as jnp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

In [3]:
flux_surface =  jax_sbgeom.flux_surfaces.FluxSurface.from_hdf5("/home/tbogaarts/stellarator_paper/base_data/vmecs/helias5_vmec.nc4")

In [4]:
tetra_mesh = jax_sbgeom.flux_surfaces.flux_surface_meshing.mesh_tetrahedra(flux_surface, jnp.linspace(0,1,10), ToroidalExtent.half_module(flux_surface), 51, 52)

face_template = jsn.tools.mesh_tools._face_template(tetra_mesh[1].shape[1])

In [5]:
normals_edges = jsn.tools.mesh_tools.create_face_normal_vectors_from_3d_simplices(tetra_mesh[0][tetra_mesh[1]], face_template)

In [ ]:
def _compute_adjacency_2(mesh_connectivity : jnp.ndarray, face_template : jnp.ndarray):
    '''
    Compute the adjacency list for a mesh.count

    See [1], pp. 428 for algorithm description. Generalised to arbitrary element types. Removed hashing in favour of lexographic sorting to avoid collisions.
    Lexographic sorting can be somewhat slow to compile though.

    Use 64-bit enabled jax to ensure you have sufficient range for face hashing. 
    For n_vertices_per_face, the maximum number of vertices is (2^63 -1)^(1/(n_vertices_per_face)) before face collisions will occur.

    [1] Hesthaven, J. S., & Warburton, T. (2008). Nodal discontinuous Galerkin methods: algorithms, analysis, and applications. New York, NY: Springer New York.

    Parameters
    ----------
    mesh_connectivity : jnp.ndarray [n_elements, n_nodes_per_element]
        The connectivity array of the mesh  
    
    Returns
    -------
    element_to_element : jnp.ndarray [n_elements, n_faces_per_element]
        The adjacency list of elements. element_to_element[i,f] gives the index of the element adjacent to element i at face f. -1 if boundary face.
    element_to_face : jnp.ndarray [n_elements, n_faces_per_element]
        The adjacency list of faces. element_to_face[i,f] gives the index of the face of the adjacent element to element i at face f. -1 if boundary face.     
    '''


    n_elements          = mesh_connectivity.shape[0]    
    n_vertices_per_face = face_template.shape[1]

    n_faces_per_element = face_template.shape[0]
    n_total_faces       = n_elements * n_faces_per_element    
    
    # for some reason, sorting using argsort and take_along_axis is faster than jnp.sort (sort takes >1s for only 100k elements, this in <<1ms)
    faces               = mesh_connectivity[:, face_template]                                # [n_elements, n_faces_per_element, n_vertices_per_face]
    faces               = jnp.take_along_axis(faces , jnp.argsort(faces, axis=-1), axis=-1)  # [n_elements, n_faces_per_element, n_vertices_per_face]
    
    faces_flat          = faces.reshape(-1, n_vertices_per_face)                              #[n_total_faces, n_vertices_per_face]
     

    # Two faces are the same if their sorted vertex indices are the same. 
    # Lexicographically sort all faces. Then the same faces will be adjacent.
    sorted_idx          = jnp.lexsort(faces_flat.T)

    # Element ids for all faces (just repeated element indices)
    elem_ids          = jnp.repeat(jnp.arange(n_elements), n_faces_per_element)   # [n_total_faces]

    # Index faces and elem_ids according to lexographic sort
    faces_flat_sorted = faces_flat[sorted_idx, :]                                 # [n_total_faces, n_vertices_per_face]
    elem_ids_sorted   = elem_ids[sorted_idx]                                      # [n_total_faces]    
    
    # Adjacent faces will be the same. Use first a mask, then convert that mask to indices. The mask will be used later as well
    
    # use -1 as the placeholder since this will always be valid (no out of bounds access later)
    same              = jnp.where(jnp.all(faces_flat_sorted[:-1,:] == faces_flat_sorted[1:, :], axis=-1), True, False)            # [n_total_faces -1] 
    same_idx          = jnp.where(same, jnp.arange(same.shape[0]), -1)                   # [n_total_faces -1]   
    
    # Left and right element of each face pair ()
    left_element     = jnp.where(same, elem_ids_sorted[:-1][same_idx], -1) # -1 placeholder
    right_element    = jnp.where(same, elem_ids_sorted[1:][same_idx] , -1) # -1 placeholder

    # Face indices within each element
    # same procedure: left and right face of each face pair (indexed within their parent element, *NOT* global face index)
    face_indices        = jnp.tile(jnp.arange(n_faces_per_element), n_elements)   # [n_total_faces]
    face_indices_sorted = face_indices[sorted_idx]

    face1 = jnp.where(same, face_indices_sorted[:-1][same_idx], -1)         # -1 placeholder 
    face2 = jnp.where(same, face_indices_sorted[1:][same_idx],  -1)         # -1 placeholder    

    # Pad to account for -1 placeholders
    # this ensures all non-connected faces can be treated in the same way as connected faces. They just set the [-1,-1] index to some meaningless value, which we will cut off at the end.
    element_to_element = jnp.full((n_elements + 1, n_faces_per_element + 1), -1)  # initialize to -1
    
    element_to_element = element_to_element.at[left_element, face1].set(right_element)
    element_to_element = element_to_element.at[right_element, face2].set(left_element)

    element_to_face    = jnp.full((n_elements + 1, n_faces_per_element + 1), -1)  # initialize to -1

    element_to_face    = element_to_face.at[left_element, face1].set(face2)
    element_to_face    = element_to_face.at[right_element, face2].set(face1)    
    
    return element_to_element[:-1, :-1], element_to_face[:-1, :-1]

jax.config.update('jax_log_compiles', True)

adj_f = jax.jit(_compute_adjacency_2)

_compute_adjacency_2(tetra_mesh[1], face_template)

(Array([[     1,     -1,      3,    150],
        [  7803,      5,      2,      0],
        [  7806,    153,      1,    151],
        ...,
        [    -1, 132646, 132649, 117044],
        [132650, 132647, 132648, 132639],
        [132649,     -1,     -1, 132640]], dtype=int64, weak_type=True),
 Array([[ 3, -1,  3,  2],
        [ 3,  3,  2,  0],
        [ 3,  1,  2,  1],
        ...,
        [-1,  2,  2,  1],
        [ 0,  3,  2,  0],
        [ 0, -1, -1,  0]], dtype=int64, weak_type=True))

In [8]:
adj_f(tetra_mesh[1], face_template)

(Array([[     1,     -1,      3,    150],
        [  7803,      5,      2,      0],
        [  7806,    153,      1,    151],
        ...,
        [    -1, 132646, 132649, 117044],
        [132650, 132647, 132648, 132639],
        [132649,     -1,     -1, 132640]], dtype=int64, weak_type=True),
 Array([[ 3, -1,  3,  2],
        [ 3,  3,  2,  0],
        [ 3,  1,  2,  1],
        ...,
        [-1,  2,  2,  1],
        [ 0,  3,  2,  0],
        [ 0, -1, -1,  0]], dtype=int64, weak_type=True))

In [9]:
%timeit adj_f(tetra_mesh[1], face_template)[1].block_until_ready()

5.37 ms ± 41.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [10]:
etoe, etof = jsn.tools.mesh_tools._compute_adjacency(tetra_mesh[1], face_template)

etoe2, etof2 = _compute_adjacency_2(tetra_mesh[1], face_template)

(530604,)
[     3    602      2 ... 529340 530577 530564]
(132651, 4, 3) (3, 4, 132651)


In [11]:
onp.testing.assert_array_equal(etoe, etoe2)
onp.testing.assert_array_equal(etof, etof2)
print(etoe2)

[[     1     -1      3    150]
 [  7803      5      2      0]
 [  7806    153      1    151]
 ...
 [    -1 132646 132649 117044]
 [132650 132647 132648 132639]
 [132649     -1     -1 132640]]


In [41]:
jnp.arange(0, -1, -1)

Array([0], dtype=int64)

In [22]:
normals_edges.shape

(132651, 4, 3)

In [15]:
plotter = pv.Plotter()
mesh = mesh_to_pyvista_mesh(tetra_mesh)
plotter.add_mesh(mesh, show_edges=True, color="white", opacity=1.0)
plotter.show()

Widget(value='<iframe src="http://localhost:38251/index.html?ui=P_0x7d942932aed0_8&reconnect=auto" class="pyvi…